Reference Code: https://github.com/umilISLab/LChange22/blob/main/cluster.py

Data set: https://www.ims.uni-stuttgart.de/en/research/resources/corpora/sem-eval-ulscd-eng/

dont need to use the lemmas??


In [ ]:
import os, zipfile, urllib.request, glob


upload the zipped data set in files

In [ ]:
DATA_DIR = './semeval2020_english'
os.makedirs(DATA_DIR, exist_ok=True)

ZIP_FILE_PATH = '/content/semeval2020_ulscd_eng.zip'
DATASET_DIR = os.path.join(DATA_DIR, 'semeval2020_ulscd_eng')

if not os.path.exists(DATASET_DIR):
    print('Unzipping English SemEval-2020 Task 1 data...')
    with zipfile.ZipFile(ZIP_FILE_PATH, 'r') as z:
        z.extractall(DATA_DIR)
    print('Done!')
else:
    print('Dataset already unzipped.')

Unzipping English SemEval-2020 Task 1 data...
Done!


load corpus and targets

In [ ]:
import re
import gzip

def load_corpus(corpus_dir):
    sentences = []
    for fpath in sorted(glob.glob(os.path.join(corpus_dir, "*.txt.gz"))): # Changed to *.txt.gz
        with gzip.open(fpath, "rt", encoding="utf-8") as f: # Use gzip.open
            sentences.extend([line.strip() for line in f if line.strip()])
    return sentences

def load_targets(dataset_dir):
    with open(os.path.join(dataset_dir, "targets.txt"), encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

def load_gold_scores(dataset_dir):
    scores = {}
    with open(os.path.join(dataset_dir, "truth", "graded.txt"), encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) == 2:
                scores[parts[0]] = float(parts[1])
    return scores

corpus1 = load_corpus(os.path.join(DATASET_DIR, "corpus1", "token"))
corpus2 = load_corpus(os.path.join(DATASET_DIR, "corpus2", "token"))
targets  = load_targets(DATASET_DIR)
gold_scores = load_gold_scores(DATASET_DIR)

# English targets have POS tags appended (e.g. "plane_nn") — strip them for word matching
target_to_word = {t: t.split("_")[0] for t in targets}

print(f"C1 sentences : {len(corpus1):,}")
print(f"C2 sentences : {len(corpus2):,}")
print(f"Target words : {len(targets)}")
print(f"\nTargets: {targets}")
print(f"\nGold scores (sample):")
for w, s in list(gold_scores.items())[:5]:
    print(f"  {w}: {s}")

C1 sentences : 253,644
C2 sentences : 353,692
Target words : 37

Targets: ['attack_nn', 'bag_nn', 'ball_nn', 'bit_nn', 'chairman_nn', 'circle_vb', 'contemplation_nn', 'donkey_nn', 'edge_nn', 'face_nn', 'fiction_nn', 'gas_nn', 'graft_nn', 'head_nn', 'land_nn', 'lane_nn', 'lass_nn', 'multitude_nn', 'ounce_nn', 'part_nn', 'pin_vb', 'plane_nn', 'player_nn', 'prop_nn', 'quilt_nn', 'rag_nn', 'record_nn', 'relationship_nn', 'risk_nn', 'savage_nn', 'stab_nn', 'stroke_vb', 'thump_nn', 'tip_vb', 'tree_nn', 'twist_nn', 'word_nn']

Gold scores (sample):
  attack_nn: 0.1439699927
  bag_nn: 0.1003636619
  ball_nn: 0.4093665525
  bit_nn: 0.3065766263
  chairman_nn: 0.0


commenting out the model stuff and loading bert

In [ ]:
# #Downloading the model from bucket

# from google_cloud_save import download_folder_from_bucket
# credentials_path = 'nlp-research-sp26-8499634f1c62.json'
# bucket_name = 'project3102-model-bucket'
# source_prefix = 'Training-Tests/McBERTh-Pretrain-100-Samples/best/'
# destination_folder = './WIDID_Model'


# download_folder_from_bucket(credentials_path, bucket_name, source_prefix, destination_folder)

BERT Embedding Extraction


*   Embeddings: sum of last 4 hidden layers
*   concatenate into one word vector



In [ ]:
# import torch
# import numpy as np
# from transformers import AutoTokenizer, AutoModel
# from tqdm import tqdm

# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# print(f"Device: {DEVICE}")

# model_path = "WIDID_Model"
# MAX_SENTENCES_PER_WORD = 300
# MAX_LENGTH = 128

# tokenizer = AutoTokenizer.from_pretrained(model_path)
# model = AutoModel.from_pretrained(model_path, output_hidden_states=True)
# model.eval().to(DEVICE)
# print("Model loaded.")

In [ ]:
import torch
import numpy as np
from transformers import BertTokenizer, BertModel
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

MAX_SENTENCES_PER_WORD = 300
MAX_LENGTH = 256 # papers orginal max legth was 256; had 128 previously

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)
model = BertModel.from_pretrained('bert-base-uncased', output_hidden_states=True)
model.eval().to(DEVICE)
print("bert-base-uncased loaded.")

Device: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


bert-base-uncased loaded.


In [ ]:
# def find_target_sentences(corpus, word):
#     pattern = re.compile(r'\b' + re.escape(word) + r'\b', re.IGNORECASE)
#     return [s for s in corpus if pattern.search(s)]
def find_target_sentences(corpus, word):

    pattern = re.compile(r'\b' + re.escape(word) + r'\w*\b', re.IGNORECASE)
    return [s for s in corpus if pattern.search(s)]


# def get_word_embedding(sentence, word, tokenizer, model, device, max_length=256):
#     """Extract summed-last-4-layer embedding for `word` in `sentence`. Returns None if not found."""
#     encoding = tokenizer(
#         sentence, return_tensors="pt", truncation=True,
#         max_length=max_length, return_offsets_mapping=True
#     )
#     offset_mapping = encoding.pop("offset_mapping")[0]

#     match = re.search(r'\b' + re.escape(word) + r'\b', sentence, re.IGNORECASE)
#     if not match:
#         return None
#     char_start, char_end = match.start(), match.end()

#     token_indices = [
#         i for i, (ts, te) in enumerate(offset_mapping.tolist())
#         if te > char_start and ts < char_end and te > ts
#     ]
#     if not token_indices:
#         return None

#     with torch.no_grad():
#         inputs = {k: v.to(device) for k, v in encoding.items()}
#         outputs = model(**inputs)

#     last4 = torch.stack(outputs.hidden_states[-4:], dim=0).sum(0)  # (1, seq, hidden)
#     return last4[0, token_indices, :].mean(0).cpu().numpy()

def get_word_embedding(sentence, word, tokenizer, model, device, max_length=256):
    encoding = tokenizer(
        sentence, return_tensors="pt", truncation=True,
        max_length=max_length, return_offsets_mapping=True
    )
    offset_mapping = encoding.pop("offset_mapping")[0]

    match = re.search(r'\b' + re.escape(word) + r'\b', sentence, re.IGNORECASE)
    if not match:
        return None
    char_start, char_end = match.start(), match.end()

    token_indices = [
        i for i, (ts, te) in enumerate(offset_mapping.tolist())
        if te > char_start and ts < char_end and te > ts
    ]
    if not token_indices:
        return None

    with torch.no_grad():
        inputs = {k: v.to(device) for k, v in encoding.items()}
        outputs = model(**inputs)

    last4 = torch.stack(outputs.hidden_states[-4:], dim=0).sum(0)  # (1, seq, hidden)

    # CONCATENATE subword tokens instead of averaging (as per the paper)
    subword_vecs = last4[0, token_indices, :]  # shape: (n_subtokens, 768)
    return subword_vecs.reshape(-1).cpu().numpy()  # concatenated: (n_subtokens * 768,)

def extract_all_embeddings(sentences, word, tokenizer, model, device,
                            max_sentences=None, max_length=128):
    if max_sentences:
        sentences = sentences[:max_sentences]
    vecs = [get_word_embedding(s, word, tokenizer, model, device, max_length)
            for s in sentences]
    vecs = [v for v in vecs if v is not None]
    return np.array(vecs) if vecs else None


print("Extraction functions defined.")

Extraction functions defined.


In [ ]:
import shutil
if os.path.exists('./embeddings_english'):
    shutil.rmtree('./embeddings_english')
    print("Cleared old embedding cache.")

EMBED_DIR = "./embeddings_english"
os.makedirs(EMBED_DIR, exist_ok=True)

embeddings_c1 = {}
embeddings_c2 = {}

for target in tqdm(targets, desc="Extracting embeddings"):
    word = target_to_word[target]
    cache_c1 = os.path.join(EMBED_DIR, f"{target}_c1.npy")
    cache_c2 = os.path.join(EMBED_DIR, f"{target}_c2.npy")

    if os.path.exists(cache_c1) and os.path.exists(cache_c2):
        embeddings_c1[target] = np.load(cache_c1)
        embeddings_c2[target] = np.load(cache_c2)
        continue

    sents1 = find_target_sentences(corpus1, word)
    sents2 = find_target_sentences(corpus2, word)

    emb1 = extract_all_embeddings(sents1, word, tokenizer, model, DEVICE,
                                   max_sentences=MAX_SENTENCES_PER_WORD)
    emb2 = extract_all_embeddings(sents2, word, tokenizer, model, DEVICE,
                                   max_sentences=MAX_SENTENCES_PER_WORD)

    if emb1 is not None and emb2 is not None:
        embeddings_c1[target] = emb1
        embeddings_c2[target] = emb2
        np.save(cache_c1, emb1)
        np.save(cache_c2, emb2)
        tqdm.write(f"  {target}: C1={emb1.shape[0]}, C2={emb2.shape[0]}")
    else:
        tqdm.write(f"  {target}: SKIPPED (no occurrences)")

print(f"\nEmbeddings ready for {len(embeddings_c1)} / {len(targets)} words.")

Extracting embeddings:   3%|▎         | 1/37 [01:10<42:30, 70.85s/it]

  attack_nn: C1=195, C2=169


Extracting embeddings:   5%|▌         | 2/37 [02:06<36:00, 61.72s/it]

  bag_nn: C1=122, C2=187


Extracting embeddings:   8%|▊         | 3/37 [02:57<32:09, 56.75s/it]

...

  tree_nn: C1=165, C2=139


Extracting embeddings:  97%|█████████▋| 36/37 [30:57<00:44, 44.64s/it]

  twist_nn: C1=115, C2=110


Extracting embeddings: 100%|██████████| 37/37 [31:44<00:00, 51.47s/it]

  word_nn: C1=130, C2=129

Embeddings ready for 37 / 37 words.


APP Clustering
Past cluster assignments are frozen, new points either join existing clusters or form new ones.

In [ ]:
from sklearn.metrics import pairwise
from sklearn.cluster import AffinityPropagation

def cosine_sim(X, Y=None):
    Y = X if Y is None else Y
    # computes cosine similarity between two sets of vectors
    # if Y is not provided, it computes similarities within X.
    return pairwise.cosine_similarity(X, Y)


class AffinityPropagationPosteriori:
    """APP — Affinity Propagation a Posteriori (WiDiD incremental clusterer)."""

    def __init__(self, trim=2, damping=0.7, max_iter=200):
        self.time_tag = 0         # tracks if its the 1st or 2nd corpus being processed
        self.trim = trim          # % threshold: clusters smaller than this % of total points are dropped
        self.damping = damping    # damping factor for Affinity Propagation (prevents oscillations aka fluctuation b/w 2 or more values)
        self.max_iter = max_iter  # max # of iterations for Affinity Propagation

        # stores the trimmed data and labels for the PREV time step (C1)
        self._prev_X_trim = None
        self._prev_labels_trim = None
        # stores the trimmed data and labels for the CURRENT time step (C2 or C1 if time_tag == 0)
        self._curr_X_trim = None
        self._curr_labels_trim = None
        # stores the data and labels after trimming for the current overall state
        self.X_ = None
        self.labels_ = None
        # store the packed (centroid) representations of clusters and their original labels
        self._X_pack = None
        self._labels_pack = None

    def _trim(self, X, labels, ignore=None):
        # filters out small clusters based on the 'trim' percentage
        # X: the data points
        # labels: the cluster assignments for each data point in X
        # ignore: optional list of cluster labels to always keep, regardless of size
        unique, counts = np.unique(labels, return_counts=True)
        # condition to keep a cluster: its count must be greater than 'trim' percent of total data points
        condition = counts > X.shape[0] * self.trim / 100
        if ignore is not None:
            # if 'ignore' labels are provided, ensure those clusters are also kept
            condition |= np.isin(unique, ignore)
        survivors = unique[condition] # get the labels of the clusters to keep
        mask = np.isin(labels, survivors) # create a boolean mask to select data points belonging to survivor clusters
        return X[mask], labels[mask] # return the trimmed data and their labels

    def _pack(self, X, labels):
        unique = np.unique(labels) # all unique cluster labels
        return np.array([X[labels == l].mean(0) for l in unique]), unique

    def fit(self, X):
        # Fits the Affinity Propagation model, either for the first corpus (C1) or incrementally for the second (C2)
        # X: the data points (embeddings) for the current corpus (C1 or C2)

        if self.time_tag == 0:
            # FIRST FIT (Processing Corpus 1 - C1)
            # Standard Affinity Propagation is applied to C1.

            ap = AffinityPropagation(damping=self.damping,
                                     affinity='precomputed',
                                     max_iter=self.max_iter,
                                     convergence_iter=30,
                                     random_state=42)

            ap.fit(cosine_sim(X)) # Fit AP using precomputed cosine similarities

            # trim the clusters based on the 'trim' percentage
            self._X_trim, self._labels_trim = self._trim(X, ap.labels_)
            # Pack the trimmed clusters into their centroid representations
            self._X_pack, self._labels_pack = self._pack(self._X_trim, self._labels_trim)

            # store the current state (C1 after trimming) for comparison in the next step
            self.X_, self.labels_ = self._X_trim, self._labels_trim

            # Fallback if AP results in very few clusters (e.g., all points in one cluster after trimming)
            if self.labels_.shape[0] <= 1:
                self.X_, self.labels_ = X, np.ones(X.shape[0], dtype=int) # Assign all points to a single cluster
                self._X_pack = X.mean(0, keepdims=True) # Create a single centroid from all points
                self._labels_pack = np.array([1])
                self._X_trim, self._labels_trim = self.X_, self.labels_

        else:
            # SECOND FIT (Processing Corpus 2 - C2 incrementally)
            # Incremental clustering: combine centroids from C1 with embeddings from C2.
            # Concatenate the packed centroids from C1 (self._X_pack) with the new data from C2 (X)
            X_next = np.concatenate([self._X_pack, X])
            # Run Affinity Propagation on this combined set (C1 centroids + C2 embeddings)
            ap = AffinityPropagation(damping=self.damping, affinity='precomputed', max_iter=self.max_iter)
            ap.fit(cosine_sim(X_next))

            # Separate the new labels for C1 centroids and C2 embeddings
            n_pack = len(self._labels_pack) # Number of C1 centroids
            new_labels_pack = ap.labels_[:n_pack] # Labels for C1 centroids in the new AP run
            labels_curr = ap.labels_[n_pack:] # Labels for C2 embeddings in the new AP run

            # Map the old C1 cluster IDs to their new IDs from the combined AP run.
            # This ensures consistency when comparing C1 and C2 clusters.
            new_labels_prev = self._labels_trim.copy()
            for old_id, new_id in zip(self._labels_pack, new_labels_pack):
                new_labels_prev[self._labels_trim == old_id] = new_id

            # Store the previous (C1) state and current (C2) state before trimming
            self._prev_X_trim = self._X_trim
            self._prev_labels_trim = new_labels_prev

            # Combine C1 (re-labeled) and C2 data and labels for final trimming
            X_all = np.concatenate([self._prev_X_trim, X])
            labels_all = np.concatenate([self._prev_labels_trim, labels_curr])
            X_all, labels_all = self._trim(X_all, labels_all) # Trim based on combined data

            # Identify which clusters survived the trimming process
            survivors = np.unique(labels_all)

            #TEMP
            # prev_mask = np.isin(self._prev_labels_trim, survivors)
            # curr_mask = np.isin(labels_curr, survivors)

            # self._prev_X_trim    = self._prev_X_trim[prev_mask]
            # self._prev_labels_trim = self._prev_labels_trim[prev_mask]
            # self._curr_X_trim    = X[curr_mask]
            # self._curr_labels_trim = labels_curr[curr_mask]


            # Filter C1 and C2 data/labels to only include points from surviving clusters
            self._prev_X_trim = self._prev_X_trim[np.isin(self._prev_labels_trim, survivors)]
            self._prev_labels_trim = self._prev_labels_trim[np.isin(self._prev_labels_trim, survivors)]
            self._curr_X_trim = X[np.isin(labels_curr, survivors)]
            self._curr_labels_trim = labels_curr[np.isin(labels_curr, survivors)]

            # Update the overall clustered data and labels
            self.X_, self.labels_ = X_all, labels_all
            # Re-pack the combined, trimmed clusters into new centroids for future potential steps (though typically only 2 steps are used here)
            self._X_pack, self._labels_pack = self._pack(X_all, labels_all)

        self.time_tag += 1 # Increment time_tag to indicate next step (or completion of second step)

print("APP class defined.")

APP class defined.


Semantic Shift Measures (JSD, PDIS, PDIV)

In [ ]:
from scipy.spatial.distance import cosine as cosine_dist
from scipy.special import rel_entr

def jsd(p, q):
    n = max(len(p), len(q))
    p = np.pad(np.array(p, float), (0, n - len(p)))
    q = np.pad(np.array(q, float), (0, n - len(q)))
    p /= p.sum() + 1e-12
    q /= q.sum() + 1e-12
    m = 0.5 * (p + q)
    return float(0.5 * rel_entr(p + 1e-12, m + 1e-12).sum() +
                 0.5 * rel_entr(q + 1e-12, m + 1e-12).sum())


def compute_scores(app):
    """Compute JSD, PDIS, PDIV from a fitted APP model (after two fit() calls)."""
    if app.time_tag < 2:
        return None

    X1, L1 = app._prev_X_trim, app._prev_labels_trim
    X2, L2 = app._curr_X_trim, app._curr_labels_trim

    if len(X1) == 0 or len(X2) == 0:
        return None

    all_clusters = np.union1d(np.unique(L1), np.unique(L2))

    # Cluster distributions
    p1 = np.array([np.sum(L1 == c) / len(L1) for c in all_clusters])
    p2 = np.array([np.sum(L2 == c) / len(L2) for c in all_clusters])
    jsd_score = jsd(p1, p2)

    # Sense prototypes
    sp1 = {c: X1[L1 == c].mean(0) for c in all_clusters if (L1 == c).sum() > 0}
    sp2 = {c: X2[L2 == c].mean(0) for c in all_clusters if (L2 == c).sum() > 0}

    if not sp1 or not sp2:
        return {"JSD": jsd_score, "PDIS": None, "PDIV": None}

    # Word prototypes (mean of sense prototypes)
    M1 = np.stack(list(sp1.values())).mean(0)
    M2 = np.stack(list(sp2.values())).mean(0)


    #check
    # print(f"  X1==X2: {np.allclose(X1, X2)}")
    # print(f"  M1==M2: {np.allclose(M1, M2)}")

    # print(f"  X1==X2: {np.allclose(X1, X2)}")
    # print(f"  M1==M2: {np.allclose(M1, M2)}")
    # print(f"  cosine(M1,M2): {cosine_dist(M1, M2):.6f}")

    # PDIS
    pdis_score = cosine_dist(M1, M2)

    # PDIV
    div1 = np.mean([cosine_dist(v, M1) for v in sp1.values()])
    div2 = np.mean([cosine_dist(v, M2) for v in sp2.values()])
    pdiv_score = abs(div1 - div2)

   # shared = np.intersect1d(np.unique(L1), np.unique(L2))
    # print(f"  L1 clusters: {np.unique(L1)}, L2 clusters: {np.unique(L2)}, shared: {shared}")
    # print(f"  sp1 keys: {sorted(np.unique(L1))}, sp2 keys: {sorted(np.unique(L2))}")

    return {"JSD": jsd_score, "PDIS": pdis_score, "PDIV": pdiv_score}


print("Shift measure functions defined.")

Shift measure functions defined.


WiDiD Pipeline

In [ ]:
# TRIM    = 2    # paper also tests 0 and 5
# DAMPING = 0.9
# MAX_ITER = 500

TRIM    = 5      # paper's best English result uses trim=5
DAMPING = 0.7    # paper's default
MAX_ITER = 500

shift_scores = {}
skipped = []

for target in tqdm(embeddings_c1, desc="WiDiD"):
    X1, X2 = embeddings_c1[target], embeddings_c2[target]
    if len(X1) < 2 or len(X2) < 2:
        skipped.append(target)
        continue
    try:
        app = AffinityPropagationPosteriori(trim=TRIM, damping=DAMPING, max_iter=MAX_ITER)
        app.fit(X1)   # t=0: cluster C1

        # print(f"  X1 shape: {X1.shape}, X2 shape: {X2.shape}")
        # print(f"{target}: C1 clusters={len(np.unique(app.labels_))}")

        app.fit(X2)   # t=1: incrementally update with C2

        #print(f"{target}: prev={len(app._prev_X_trim)}, curr={len(app._curr_X_trim) if app._curr_X_trim is not None else 'None'}")

        s = compute_scores(app)
        if s:
            shift_scores[target] = s
        else:
            skipped.append(target)
    except Exception as e:
        tqdm.write(f"  {target}: ERROR — {e}")
        skipped.append(target)

print(f"\nScored {len(shift_scores)} words. Skipped: {skipped}")
print("\nSample:")
for t, s in list(shift_scores.items())[:5]:
    #print(f"  {t}: JSD={s['JSD']:.4f}  PDIS={s.get('PDIS', 'N/A'):.4f}  PDIV={s.get('PDIV', 'N/A'):.4f}")

    print(f"  {t}: JSD={s['JSD']:.6f}  PDIS={s['PDIS']:.12f}  PDIV={s['PDIV']:.12f}")
print("\nDone.")

WiDiD: 100%|██████████| 37/37 [00:02<00:00, 15.82it/s]


Scored 37 words. Skipped: []

Sample:
  attack_nn: JSD=0.149166  PDIS=0.067566812038  PDIV=0.066048681736
  bag_nn: JSD=0.033805  PDIS=0.027658402920  PDIV=0.001000303775
  ball_nn: JSD=0.142786  PDIS=0.051283240318  PDIV=0.011743724346
  bit_nn: JSD=0.117719  PDIS=0.050853133202  PDIV=0.008953005075
  chairman_nn: JSD=0.123146  PDIS=0.049079954624  PDIV=0.008446872234

Done.


In [ ]:
# all samples
for t, s in shift_scores.items():
    print(f"{t}: JSD={s['JSD']:.6f}  PDIS={s['PDIS']:.12f}  PDIV={s['PDIV']:.12f}")

attack_nn: JSD=0.149166  PDIS=0.067566812038  PDIV=0.066048681736
bag_nn: JSD=0.033805  PDIS=0.027658402920  PDIV=0.001000303775
ball_nn: JSD=0.142786  PDIS=0.051283240318  PDIV=0.011743724346
bit_nn: JSD=0.117719  PDIS=0.050853133202  PDIV=0.008953005075
chairman_nn: JSD=0.123146  PDIS=0.049079954624  PDIV=0.008446872234
circle_vb: JSD=0.040475  PDIS=0.019088745117  PDIV=0.013440623879
contemplation_nn: JSD=0.029515  PDIS=0.014983475208  PDIV=0.000671645626
donkey_nn: JSD=0.000512  PDIS=0.010329723358  PDIV=0.001861453056
edge_nn: JSD=0.002194  PDIS=0.013177096844  PDIV=0.001653719693
face_nn: JSD=0.027485  PDIS=0.020798683167  PDIV=0.014829181135
fiction_nn: JSD=0.050193  PDIS=0.022985398769  PDIV=0.005947880447
gas_nn: JSD=0.178835  PDIS=0.080071806908  PDIV=0.048273004591
graft_nn: JSD=0.366174  PDIS=0.101719319820  PDIV=0.062469504774
head_nn: JSD=0.137885  PDIS=0.039413571358  PDIV=0.024901255965
land_nn: JSD=0.026733  PDIS=0.021565854549  PDIV=0.015162386000
lane_nn: JSD=0.15420

In [ ]:
# scores ranked by JSD from highest to lowest
sorted_scores = sorted(shift_scores.items(), key=lambda x: x[1]['JSD'], reverse=True)
for t, s in sorted_scores:
    print(f"{t}: JSD={s['JSD']:.6f}  PDIS={s['PDIS']:.12f}  PDIV={s['PDIV']:.12f}")

plane_nn: JSD=0.531359  PDIS=0.252571642399  PDIV=0.066394142807
rag_nn: JSD=0.414565  PDIS=0.080527782440  PDIV=0.097160100937
graft_nn: JSD=0.366174  PDIS=0.101719319820  PDIV=0.062469504774
ounce_nn: JSD=0.262290  PDIS=0.060659050941  PDIV=0.051730260253
stab_nn: JSD=0.248797  PDIS=0.076700150967  PDIV=0.063904993236
player_nn: JSD=0.243223  PDIS=0.085626721382  PDIV=0.006276793778
multitude_nn: JSD=0.223052  PDIS=0.016371250153  PDIV=0.007071517408
stroke_vb: JSD=0.204831  PDIS=0.065351486206  PDIV=0.075298443437
part_nn: JSD=0.179257  PDIS=0.049439787865  PDIV=0.016066804528
gas_nn: JSD=0.178835  PDIS=0.080071806908  PDIV=0.048273004591
tip_vb: JSD=0.172937  PDIS=0.057700514793  PDIV=0.017241999507
lane_nn: JSD=0.154202  PDIS=0.076321840286  PDIV=0.058050967753
attack_nn: JSD=0.149166  PDIS=0.067566812038  PDIV=0.066048681736
ball_nn: JSD=0.142786  PDIS=0.051283240318  PDIV=0.011743724346
head_nn: JSD=0.137885  PDIS=0.039413571358  PDIV=0.024901255965
lass_nn: JSD=0.128292  PDIS=0

In [ ]:
sorted_scores = sorted(shift_scores.items(), key=lambda x: x[1]['JSD'], reverse=True)
print(f"{'word':<20} {'JSD':>10} {'gold':>10}")
print("-" * 42)
for t, s in sorted_scores:
    gold = gold_scores.get(t, "N/A")
    print(f"{t:<20} {s['JSD']:>10.6f} {gold:>10.4f}")

word                        JSD       gold
------------------------------------------
plane_nn               0.531359     0.8823
rag_nn                 0.414565     0.2765
graft_nn               0.366174     0.5540
ounce_nn               0.262290     0.2849
stab_nn                0.248797     0.4006
player_nn              0.243223     0.2737
multitude_nn           0.223052     0.1004
stroke_vb              0.204831     0.1762
part_nn                0.179257     0.1613
gas_nn                 0.178835     0.1596
tip_vb                 0.172937     0.6789
lane_nn                0.154202     0.1037
attack_nn              0.149166     0.1440
ball_nn                0.142786     0.4094
head_nn                0.137885     0.2953
lass_nn                0.128292     0.2126
quilt_nn               0.126866     0.1231
chairman_nn            0.123146     0.0000
bit_nn                 0.117719     0.3066
thump_nn               0.084142     0.1430
risk_nn                0.073486     0.0000
relationshi

In [ ]:
import pandas as pd

# Build comparison dataframe
words = [t for t in shift_scores if t in gold_scores]

df = pd.DataFrame({
    'word': words,
    'JSD': [shift_scores[t]['JSD'] for t in words],
    'gold': [gold_scores[t] for t in words]
})

# Add rankings (1 = highest/most change)
df['JSD_rank'] = df['JSD'].rank(ascending=False).astype(int)
df['gold_rank'] = df['gold'].rank(ascending=False).astype(int)
df['rank_diff'] = abs(df['JSD_rank'] - df['gold_rank'])

# Sort by gold rank
df = df.sort_values('gold_rank')

print(f"{'word':<20} {'gold':>8} {'gold_rank':>10} {'JSD':>10} {'JSD_rank':>10} {'|diff|':>8}")
print("-" * 70)
for _, row in df.iterrows():
    print(f"{row['word']:<20} {row['gold']:>8.4f} {row['gold_rank']:>10} {row['JSD']:>10.6f} {row['JSD_rank']:>10} {row['rank_diff']:>8}")

word                     gold  gold_rank        JSD   JSD_rank   |diff|
----------------------------------------------------------------------
plane_nn               0.8823          1   0.531359          1        0
tip_vb                 0.6789          2   0.172937         11        9
prop_nn                0.6248          3   0.061555         25       22
graft_nn               0.5540          4   0.366174          3        1
record_nn              0.4274          5   0.034099         29       24
ball_nn                0.4094          6   0.142786         14        8
stab_nn                0.4006          7   0.248797          5        2
twist_nn               0.3985          8   0.061628         24       16
bit_nn                 0.3066          9   0.117719         19       10
head_nn                0.2953         10   0.137885         15        5
ounce_nn               0.2849         11   0.262290          4        7
rag_nn                 0.2765         12   0.414565          2   

In [ ]:
from scipy import stats

words = [t for t in shift_scores if t in gold_scores]
predicted = [shift_scores[t]['JSD'] for t in words]
gold = [gold_scores[t] for t in words]

r, p = stats.spearmanr(predicted, gold)
print(f"Spearman r = {r:.3f}  (p={p:.3f})")


Spearman r = 0.371  (p=0.024)
